Prompt evaluation


Prompt engineering is your toolkit for crafting effective prompts. It includes techniques like:

Multishot prompting
Structuring with XML tags
Many other best practices
These techniques help Claude understand exactly what you're asking for and how you want it to respond.

Prompt evaluation takes a different approach. Instead of focusing on how to write prompts, it's about measuring their effectiveness through automated testing. You can:

Test against expected answers
Compare different versions of the same prompt
Review outputs for errors

![](../images/img13.png)
![](../images/img14.png)




# A Typical Prompt Evaluation Workflow

A typical prompt evaluation workflow follows five key steps that help you systematically improve prompts through objective measurement.

While there are many ways to design these workflows (and many tools available), the core idea remains the same: start small, measure performance, and iterate.

---

# Step 1: Draft a Prompt

Start by writing an initial prompt that you want to improve.

Example:

```python
prompt = f"""
Please answer the user's question:

{question}
"""
```

This serves as your **baseline prompt** for evaluation.

---

# Step 2: Create an Eval Dataset

Your evaluation dataset contains sample inputs that represent real-world usage.

For example:

- "What's 2+2?"
- "How do I make oatmeal?"
- "How far away is the Moon?"

In real applications, this dataset can include:

- tens
- hundreds
- or even thousands of examples

These datasets can be manually created or generated using models like Claude.

---

# Step 3: Feed Through Claude

Each dataset question is inserted into the prompt template.

Example:

```text
Please answer the user's question:
What's 2+2?
```

Claude then generates responses such as:

- "2 + 2 = 4"
- oatmeal instructions
- distance to the Moon

---

# Step 4: Feed Through a Grader

A grader evaluates the quality of Claude’s responses by analyzing:

- the original question
- the generated answer

It assigns a score (typically 1 to 10):

- 10 = perfect answer
- lower scores = needs improvement

---

### Example Scores

- Math question → 10  
- Oatmeal question → 4  
- Moon question → 9  

Average score:

$$
\frac{10 + 4 + 9}{3} = 7.66
$$

---

# Step 5: Change Prompt and Repeat

Now improve the prompt based on evaluation results.

Example:

```python
prompt = f"""
Please answer the user's question:

{question}

Answer the question with ample detail.
"""
```

After re-running the evaluation, the new average score might improve (e.g., 8.7), showing that the prompt has improved.

---

# Prompt Scoring

The key benefit of this workflow is **objective evaluation** of prompt quality.

You can:

- Compare prompt versions numerically
- Select the best-performing prompt
- Continuously iterate for improvement

---

# Summary

This workflow transforms prompt engineering into a measurable process:

1. Write prompt  
2. Build dataset  
3. Generate responses  
4. Grade outputs  
5. Improve and repeat  

This removes guesswork and ensures each change is a real improvement.

In [96]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv(override=True) 
API_KEY = os.getenv("GEMINI_API_KEY") 

client  = genai.Client(api_key=API_KEY)

In [97]:
# Helper functions
def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "parts": [{"text": text}]
    })

def add_model_message(messages, text):
    messages.append({
        "role": "model",
        "parts": [{"text": text}]
    })

def chat(messages, response_schema = None, system_prompt=None, temperature=1.0):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=messages,
        config=genai.types.GenerateContentConfig(
            system_instruction=system_prompt,
            max_output_tokens=4000,
            temperature=temperature,
            response_schema=response_schema,
            response_mime_type="application/json"
        )
    )

    return response.text

In [98]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks.

Rules:
- Each task must be solvable with a single Python function, a single JSON object, or a regex
- Keep tasks simple and atomic
- Generate exactly 3 items
"""

    messages = []

    schema = {
        "type": "array",
        "items": {
            "type": "object",
            "properties": {
                "task":   {"type": "string"},
                "format": {"type": "string", "enum": ["python", "json", "regex"]}  # 👈 added format field
            },
            "required": ["task", "format"]
        }
    }

    add_user_message(messages, prompt)

    text = chat(messages, schema)

    return json.loads(text)

In [99]:
dataset = generate_dataset()
dataset

[{'task': 'Write a Python function to list all S3 bucket names in an AWS account.',
  'format': 'python'},
 {'task': 'Generate an IAM policy that allows read-only access to all S3 buckets.',
  'format': 'json'},
 {'task': 'Create a regular expression to validate an AWS S3 bucket ARN.',
  'format': 'regex'}]

In [100]:

with open("dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

Now that we have our evaluation dataset ready, it's time to build the core evaluation pipeline. This involves taking each test case, merging it with our prompt, feeding it to Claude, and then grading the results.

In [101]:
schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "task": {"type": "string"}
        },
        "required": ["task"]
    }
}

def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments, commentary, or explanation
* Do not wrap in markdown or backticks
* Output the raw code only
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)  
    return output

In [102]:
def grade_by_model(test_case, output):
    eval_schema = {                     
        "type": "object",
        "properties": {
            "strengths":  {"type": "array", "items": {"type": "string"}},
            "weaknesses": {"type": "array", "items": {"type": "string"}},
            "reasoning":  {"type": "string"},
            "score":      {"type": "integer"}
        }
    }

    eval_prompt = f"""
        You are an expert AWS code reviewer. Evaluate the following AI-generated solution.

        Original Task:
        <task>{test_case["task"]}</task>

        Solution to Evaluate:
        <solution>{output}</solution>

        Provide strengths, weaknesses, reasoning, and a score from 1-10.
    """
    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, eval_schema)   
    return json.loads(eval_text)



In [103]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [104]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    syntax_score = grade_syntax(output, test_case)
    score = (model_grade["score"] + syntax_score) / 2
    return {
        "output":    output,
        "test_case": test_case,
        "score":     model_grade["score"],
        "reasoning": model_grade["reasoning"]
    }

In [105]:
from statistics import mean

def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    return results    

In [106]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

for r in results:
    print(f"Task:      {r['test_case']['task']}")
    print(f"Score:     {r['score']}/10")
    print(f"Reasoning: {r['reasoning']}")
    print("-" * 50)

Average Score: 6
Task:      Write a Python function to list all S3 bucket names in an AWS account.
Score:     9/10
Reasoning: The solution effectively solves the task of listing all S3 bucket names. It utilizes `boto3` correctly and includes essential elements like a docstring and a demonstration block. The error handling is present, but its generic nature and console-only output for errors are areas for improvement. For `list_buckets`, pagination is not typically a concern as S3 generally returns all buckets in a single response. Overall, it's a solid, functional piece of code, but could be made more robust in its error management for a production setting.
--------------------------------------------------
Task:      Generate an IAM policy that allows read-only access to all S3 buckets.
Score:     3/10
Reasoning: The solution demonstrates a good understanding of the necessary actions for S3 read-only access, correctly separating account-level, bucket-level, and initiating object-level